# OmniVoice Quick Start

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/k2-fsa/OmniVoice/blob/master/docs/OmniVoice.ipynb)

This notebook demonstrates the basic usage of [OmniVoice](https://github.com/k2-fsa/OmniVoice), a massively multilingual zero-shot TTS model supporting 600+ languages.

**Contents:**
1. Installation
2. Option A — Gradio Demo (interactive web UI, no code needed)
3. Option B — Python API
   - 3.1 Load Model
   - 3.2 Voice Cloning
   - 3.3 Voice Design
   - 3.4 Auto Voice

## 1. Installation

Colab already provides a compatible PyTorch + CUDA environment, so we only need to install OmniVoice.

In [1]:
!pip install omnivoice

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.2/160.2 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 4.1 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [13]:
!pip install Flask
!pip install flask_ngrok

In [ ]:
from flask import Flask, request, jsonify
import io
import base64
import soundfile as sf
from flask_ngrok import run_with_ngrok

# Ensure the model is loaded before running the Flask app
# If you restart your kernel, please re-run the '3.1 Load Model' cell (RfJXQ6zbCpTX)
# to ensure the 'model' object is available.

app = Flask(__name__)
run_with_ngrok(app) # Starts ngrok when app is run

@app.route('/generate_audio', methods=['POST'])
def generate_audio_api():
    data = request.json
    text = data.get('text')
    ref_audio_base64 = data.get('ref_audio_base64') # Optional: base64 encoded audio
    instruct = data.get('instruct') # Optional: for voice design

    if not text:
        return jsonify({"error": "'text' parameter is required."}), 400

    generate_kwargs = {"text": text}

    ref_audio_path = None
    if ref_audio_base64:
        try:
            # Decode base64 audio and save to a temporary file
            audio_bytes = base64.b64decode(ref_audio_base64)
            ref_audio_path = "temp_ref_audio.wav"
            with open(ref_audio_path, "wb") as f:
                f.write(audio_bytes)
            generate_kwargs["ref_audio"] = ref_audio_path
        except Exception as e:
            return jsonify({"error": f"Invalid ref_audio_base64: {e}"}), 400

    if instruct:
        generate_kwargs["instruct"] = instruct

    try:
        audio = model.generate(**generate_kwargs)

        # Convert generated audio to base64 string
        buffer = io.BytesIO()
        sf.write(buffer, audio[0], 24000, format='WAV')
        audio_base64 = base64.b64encode(buffer.getvalue()).decode('utf-8')

        return jsonify({"audio_base64": audio_base64}), 200
    except Exception as e:
        return jsonify({"error": f"Audio generation failed: {e}"}), 500

if __name__ == '__main__':
    # The app.run() call is handled by run_with_ngrok()
    app.run()

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
Exception in thread Thread-5:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connection.py", line 198, in _new_conn
    sock = connection.create_connection(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/urllib3/util/connection.py", line 85, in create_connection
    raise err
  File "/usr/local/lib/python3.12/dist-packages/urllib3/util/connection.py", line 73, in create_connection
    sock.connect(sa)
ConnectionRefusedError: [Errno 111] Connection refused

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py", line 787, in urlopen
    response = self._make_request(
            

### How to use this Flask API in Colab:

1.  **Run all previous cells**: Make sure you have run the installation cells, especially `!pip install omnivoice` and the `3.1 Load Model` cell (`RfJXQ6zbCpTX`) to load the `model` object.
2.  **Run the Flask app cell**: Execute the Python cell above containing the Flask application code.
3.  **Accessing the API**:
    *   **Within Colab (local)**: The app will be running on `http://0.0.0.0:5000`. You can make requests from other cells in this notebook (e.g., using `requests` library) to `http://localhost:5000/generate_audio`.
    *   **From your external web app (public)**: Directly accessing `http://0.0.0.0:5000` from outside Colab is not possible. You need a tunneling service like `ngrok` or `flask-ngrok` to get a public URL. You would install `ngrok` (e.g., `!pip install flask-ngrok`) and then wrap `app.run()` with `run_with_ngrok(app)` to get a public URL.

### Example API Call (Python in another Colab cell):

```python
import requests
import base64

# Assuming the Flask app is running on port 5000 (local to Colab)
# If using ngrok, replace with your ngrok public URL
API_URL = "http://localhost:5000/generate_audio"

# Example 1: Text-to-Speech with Auto Voice
payload = {
    "text": "This is an example text for the API."
}
response = requests.post(API_URL, json=payload)

if response.status_code == 200:
    result = response.json()
    audio_data = base64.b64decode(result["audio_base64"])
    with open("api_output_auto.wav", "wb") as f:
        f.write(audio_data)
    print("Audio saved to api_output_auto.wav")
else:
    print(f"API Error: {response.status_code} - {response.json().get('error')}")

# Example 2: Text-to-Speech with Voice Design
payload_design = {
    "text": "Hello, I am a designed voice from the API.",
    "instruct": "male, high pitch, American accent"
}
response_design = requests.post(API_URL, json=payload_design)

if response_design.status_code == 200:
    result_design = response_design.json()
    audio_data_design = base64.b64decode(result_design["audio_base64"])
    with open("api_output_design.wav", "wb") as f:
        f.write(audio_data_design)
    print("Audio saved to api_output_design.wav")
else:
    print(f"API Error: {response_design.status_code} - {response_design.json().get('error')}")

# Example 3: Text-to-Speech with Voice Cloning (requires a base64 encoded audio file)
# For demonstration, let's use a dummy base64 string for 'ref_audio_base64'.
# In a real scenario, you would read your ref_audio.wav, encode it to base64, and send it.
# with open("your_ref_audio.wav", "rb") as f:
#     ref_audio_bytes = f.read()
# ref_audio_base64_str = base64.b64encode(ref_audio_bytes).decode('utf-8')

# For simplicity, using a previously uploaded audio file and encoding it on the fly
try:
    with open("audio (3).wav", "rb") as f:
        ref_audio_bytes = f.read()
    ref_audio_base64_str = base64.b64encode(ref_audio_bytes).decode('utf-8')

    payload_clone = {
        "text": "This is a cloned voice.",
        "ref_audio_base64": ref_audio_base64_str
    }
    response_clone = requests.post(API_URL, json=payload_clone)

    if response_clone.status_code == 200:
        result_clone = response_clone.json()
        audio_data_clone = base64.b64decode(result_clone["audio_base64"])
        with open("api_output_clone.wav", "wb") as f:
            f.write(audio_data_clone)
        print("Audio saved to api_output_clone.wav")
    else:
        print(f"API Error: {response_clone.status_code} - {response_clone.json().get('error')}")
except FileNotFoundError:
    print("Reference audio 'audio (3).wav' not found. Please ensure it's uploaded or provide a valid path.")

```

### Set up ngrok for public access

To expose your Flask app to the internet, we'll use `ngrok`. You'll need to set your ngrok authentication token.

1.  **Add your ngrok API key to Colab secrets**: Click the "🔑" icon in the left sidebar, then click "Add new secret". Name the secret `NGROK_AUTH_TOKEN` and paste your ngrok authentication token as its value.
2.  **Enable access**: Make sure "Notebook access" is enabled for `NGROK_AUTH_TOKEN`.

After setting up the secret, the next cell will install `flask-ngrok` and `pyngrok` and configure `ngrok` using your token.

In [ ]:
!pip install flask-ngrok pyngrok

from google.colab import userdata
import os

# Get ngrok token from Colab secrets
NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')

# Authenticate ngrok
if NGROK_AUTH_TOKEN:
    os.environ["NGROK_AUTHTOKEN"] = NGROK_AUTH_TOKEN
    print("ngrok authentication token set from Colab secrets.")
else:
    print("Warning: NGROK_AUTH_TOKEN not found in Colab secrets. Please add it to secrets for public access.")

## 2. Option A — Gradio Demo

Launch an interactive web UI with a public Gradio link. The `--share` flag creates a temporary public URL so you can access the demo from any browser.

> **If you prefer to use the Python API directly, skip to Option B below.**

In [2]:
!omnivoice-demo --share

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):
2026-04-23 15:58:55,774 root INFO: Loading model from k2-fsa/OmniVoice, device=cuda ...
2026-04-23 15:58:55,928 huggingface_hub.utils._http WARNING: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
config.json: 2.24kB [00:00, 1.86MB/s]
model.safetensors: 100% 2.

## 3. Option B — Python API

### 3.1 Load Model

In [3]:
from omnivoice import OmniVoice
import soundfile as sf
import torch
from IPython.display import Audio, display

model = OmniVoice.from_pretrained(
    "k2-fsa/OmniVoice",
    device_map="cuda:0",
    dtype=torch.float16,
    load_asr=True,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/313 [00:00<?, ?it/s]

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/527 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

### 3.2 Voice Cloning

Clone a voice from a short (3-10s) reference audio clip. Upload your own `ref.wav` or use any audio file.

`ref_text` is optional — if omitted, the model uses Whisper ASR to auto-transcribe it.

In [4]:
from google.colab import files

print("Upload a reference audio file (wav/mp3/flac):")
uploaded = files.upload()
ref_audio_path = list(uploaded.keys())[0]
print(f"Uploaded: {ref_audio_path}")

Upload a reference audio file (wav/mp3/flac):


Saving audio (3).wav to audio (3).wav
Uploaded: audio (3).wav


In [5]:
audio = model.generate(
    text="Hello, this is a test of zero-shot voice cloning.",
    ref_audio=ref_audio_path,
    # ref_text="Transcription of the reference audio.",  # optional
)

sf.write("clone_out.wav", audio[0], 24000)
display(Audio(audio[0], rate=24000))

[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see re

### 3.3 Voice Design

Describe the desired voice with speaker attributes — no reference audio needed.

Supported attributes: gender, age, pitch, style (whisper), English accent, Chinese dialect. See [docs/voice-design.md](https://github.com/k2-fsa/OmniVoice/blob/master/docs/voice-design.md) for the full list.

In [8]:
audio = model.generate(
    text="Hello, this is a test of zero-shot voice design.",
    instruct="female, low pitch, british accent",
)

sf.write("design_out.wav", audio[0], 24000)
display(Audio(audio[0], rate=24000))

### 3.4 Auto Voice

Let the model choose a voice automatically — no reference audio or instruct needed.

In [7]:
audio = model.generate(
    text="This is a sentence generated with automatic voice selection.",
)

sf.write("auto_out.wav", audio[0], 24000)
display(Audio(audio[0], rate=24000))